<a href="https://colab.research.google.com/github/hamzaqarni1/DeepLearning/blob/main/Tutorial_14/Tutorial_14_A_RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from google.colab import files
import matplotlib.colors as mcolors

# --- Set Seed for Reproducibility ---
torch.manual_seed(42)

# ==============================================================================
# 1. CUSTOM DATASET PREPARATION
# ==============================================================================
sentences = [
    "Elon Musk founded SpaceX in California",
    "Tim Cook leads Apple in Cupertino",
    "Bill Gates created Microsoft in Washington",
    "Sundar Pichai works at Google in California",
    "Satya Nadella runs Alphabet in America"
]

labels = [
    ["PERSON", "PERSON", "O", "ORGANIZATION", "O", "LOCATION"],
    ["PERSON", "PERSON", "O", "ORGANIZATION", "O", "LOCATION"],
    ["PERSON", "PERSON", "O", "ORGANIZATION", "O", "LOCATION"],
    ["PERSON", "PERSON", "O", "O", "ORGANIZATION", "O", "LOCATION"],
    ["PERSON", "PERSON", "O", "ORGANIZATION", "O", "LOCATION"]
]

word2idx = {"<PAD>": 0, "<UNK>": 1}
label2idx = {"<PAD>": 0, "O": 1, "PERSON": 2, "ORGANIZATION": 3, "LOCATION": 4}

for sentence in sentences:
    for word in sentence.split():
        if word not in word2idx:
            word2idx[word] = len(word2idx)

idx2label = {v: k for k, v in label2idx.items()}

def encode_sequence(seq, mapping):
    return [mapping.get(item, mapping.get("<UNK>", 0)) for item in seq]

X = [encode_sequence(s.split(), word2idx) for s in sentences]
y = [encode_sequence(l, label2idx) for l in labels]

max_len = max(len(s) for s in X)
X_padded = [s + [word2idx["<PAD>"]] * (max_len - len(s)) for s in X]
y_padded = [l + [label2idx["<PAD>"]] * (max_len - len(l)) for l in y]

X_tensor = torch.tensor(X_padded, dtype=torch.long)
y_tensor = torch.tensor(y_padded, dtype=torch.long)

# ==============================================================================
# 2. ARCHITECTURE DEFINITION
# ==============================================================================
class SimpleRNN_NER(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super(SimpleRNN_NER, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embeds = self.embedding(x)
        out, _ = self.rnn(embeds)
        out = self.dropout(out)
        out = self.fc(out)
        return out

VOCAB_SIZE = len(word2idx)
EMBED_DIM = 50
NUM_CLASSES = len(label2idx)

def train_model(hidden_dim, learning_rate, epochs, name):
    print(f"\n--- Training {name} (Units: {hidden_dim}, LR: {learning_rate}, Epochs: {epochs}) ---")
    model = SimpleRNN_NER(VOCAB_SIZE, EMBED_DIM, hidden_dim, NUM_CLASSES)
    criterion = nn.CrossEntropyLoss(ignore_index=0)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    loss_history = []
    model.train()
    for epoch in range(epochs):
        optimizer.zero_grad()
        outputs = model(X_tensor)
        loss = criterion(outputs.view(-1, NUM_CLASSES), y_tensor.view(-1))
        loss.backward()
        optimizer.step()

        loss_history.append(loss.item())
        if (epoch + 1) % max(1, epochs // 5) == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}")

    return model, loss_history

# ==============================================================================
# 3. HYPERPARAMETER LOOP TRAINING (Task 2)
# ==============================================================================
# Defining 3 different configurations
configs = [
    {"name": "Baseline", "units": 50, "lr": 0.001, "epochs": 15, "color": "blue"},
    {"name": "Moderate", "units": 80, "lr": 0.005, "epochs": 30, "color": "orange"},
    {"name": "Aggressive", "units": 128, "lr": 0.01, "epochs": 50, "color": "red"}
]

models = {}
histories = {}
final_losses = []

# LOOP OVER CONFIGS
for cfg in configs:
    m, h = train_model(cfg["units"], cfg["lr"], cfg["epochs"], cfg["name"])
    models[cfg["name"]] = m
    histories[cfg["name"]] = h
    final_losses.append(h[-1])

# Extract the best model (Aggressive) for later visual inference
best_model = models["Aggressive"]

# ==============================================================================
# 4. VISUAL 1: MULTI-MODEL LOSS COMPARISON CURVE
# ==============================================================================
plt.figure(figsize=(10, 5))
for cfg in configs:
    name = cfg["name"]
    plt.plot(range(1, cfg["epochs"] + 1), histories[name], color=cfg["color"], linewidth=2.5,
             label=f'{name} (Units: {cfg["units"]}, LR: {cfg["lr"]})')

plt.title("Task 2: Hyperparameter Optimization Convergence", fontweight='bold', fontsize=14)
plt.xlabel("Epochs Trained", fontweight='bold')
plt.ylabel("CrossEntropy Loss", fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.savefig("rnn_multi_loss_curve.png", dpi=300)
plt.close()
files.download("rnn_multi_loss_curve.png")

# ==============================================================================
# 5. VISUAL 2: FINAL LOSS BAR CHART
# ==============================================================================
plt.figure(figsize=(8, 4.5))
bars = plt.bar([cfg["name"] for cfg in configs], final_losses, color=[cfg["color"] for cfg in configs], alpha=0.8)
plt.title("Final Terminal Loss by Model Configuration", fontweight='bold', fontsize=14)
plt.ylabel("Final CrossEntropy Loss", fontweight='bold')

# Add text labels on top of bars
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.01, f'{yval:.4f}', ha='center', va='bottom', fontweight='bold')

plt.ylim(0, max(final_losses) + 0.5)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("rnn_final_loss_bar.png", dpi=300)
plt.close()
files.download("rnn_final_loss_bar.png")

# ==============================================================================
# 6. VISUAL 3: TESTING & GRAPHICAL INFERENCE TABLE
# ==============================================================================
print("\n--- Testing Custom Sentence on Best Model ---")
best_model.eval()

test_sentence = "Mark Zuckerberg built Facebook in California"
test_seq = encode_sequence(test_sentence.split(), word2idx)
test_seq_padded = test_seq + [word2idx["<PAD>"]] * (max_len - len(test_seq))
test_tensor = torch.tensor([test_seq_padded], dtype=torch.long)

with torch.no_grad():
    raw_logits = best_model(test_tensor)
    predictions = torch.softmax(raw_logits, dim=-1).squeeze().numpy()
    predicted_indices = np.argmax(predictions, axis=-1).tolist()

words = test_sentence.split()
pred_labels = [idx2label.get(idx, "O") for idx in predicted_indices[:len(words)]]

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('tight')
ax.axis('off')
color_map = {"PERSON": "#d4edda", "ORGANIZATION": "#cce5ff", "LOCATION": "#f8d7da", "O": "#f2f2f2"}
cell_colors = [["#ffffff", color_map.get(label, "#ffffff")] for label in pred_labels]
table_data = [[w, l] for w, l in zip(words, pred_labels)]

table = ax.table(cellText=table_data, colLabels=['Input Token', 'Predicted NER Entity'],
                 loc='center', cellColours=cell_colors, cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1, 2)
for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#343a40')

plt.title("NER Inference on Unseen Custom Sequence (Best Model)", fontweight='bold', fontsize=14, pad=20)
plt.tight_layout()
plt.savefig("ner_prediction_table.png", dpi=300, bbox_inches='tight')
plt.close()
files.download("ner_prediction_table.png")

# ==============================================================================
# 7. VISUAL 4: NER PROBABILITY HEATMAP
# ==============================================================================
word_probs = predictions[:len(words)]
class_labels = [idx2label[i] for i in range(NUM_CLASSES)]

plt.figure(figsize=(9, 5))
plt.imshow(word_probs, cmap='Blues', aspect='auto')

plt.xticks(ticks=np.arange(NUM_CLASSES), labels=class_labels, rotation=45, ha='right', fontweight='bold')
plt.yticks(ticks=np.arange(len(words)), labels=words, fontweight='bold')

for i in range(len(words)):
    for j in range(NUM_CLASSES):
        text_color = "white" if word_probs[i, j] > 0.5 else "black"
        plt.text(j, i, f"{word_probs[i, j]:.2f}", ha="center", va="center", color=text_color)

plt.title("Softmax Confidence Heatmap for NER Predictions", fontweight='bold', fontsize=14)
plt.colorbar(label='Probability Confidence')
plt.tight_layout()
plt.savefig("ner_probability_heatmap.png", dpi=300)
plt.close()
files.download("ner_probability_heatmap.png")

# ==============================================================================
# 8. VISUAL 5: WORD EMBEDDING PCA CLUSTERING
# ==============================================================================
embeddings = best_model.embedding.weight.detach().numpy()
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

plt.figure(figsize=(10, 7))
plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], color='mediumseagreen', edgecolors='black', s=100)

for word, idx in word2idx.items():
    if word not in ["<PAD>", "<UNK>"]:
        plt.annotate(word, (embeddings_2d[idx, 0] + 0.05, embeddings_2d[idx, 1] + 0.05), fontsize=11)

plt.title("PCA Projection of Learned Word Embeddings", fontweight='bold', fontsize=14)
plt.xlabel("Principal Component 1", fontweight='bold')
plt.ylabel("Principal Component 2", fontweight='bold')
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("word_embedding_pca.png", dpi=300)
plt.close()
files.download("word_embedding_pca.png")

# ==============================================================================
# 9. VISUAL 6: HYPERPARAMETER SUMMARY TABLE
# ==============================================================================
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.axis('tight')
ax.axis('off')

summary_data = []
for cfg, f_loss in zip(configs, final_losses):
    summary_data.append([cfg['name'], cfg['units'], cfg['epochs'], cfg['lr'], f"{f_loss:.4f}"])

col_labels = ['Model Config', 'Hidden Units', 'Epochs', 'Learning Rate', 'Terminal Loss']
table = ax.table(cellText=summary_data, colLabels=col_labels, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)

for (row, col), cell in table.get_celld().items():
    if row == 0:
        cell.set_text_props(weight='bold', color='white')
        cell.set_facecolor('#2c3e50')
    elif col == 4 and row > 0:
        cell.set_text_props(weight='bold')
        if "Aggressive" in summary_data[row-1][0]:
            cell.set_facecolor('#d4edda') # Highlight best loss

plt.title("Hyperparameter Configuration Matrix", fontweight='bold', fontsize=14, pad=15)
plt.tight_layout()
plt.savefig("hyperparameter_summary.png", dpi=300, bbox_inches='tight')
plt.close()
files.download("hyperparameter_summary.png")

print("Execution Complete! Downloaded 6 high-quality analytical figures for your report.")


--- Training Baseline (Units: 50, LR: 0.001, Epochs: 15) ---
Epoch 3/15 | Loss: 1.4901
Epoch 6/15 | Loss: 1.3295
Epoch 9/15 | Loss: 1.1932
Epoch 12/15 | Loss: 1.1094
Epoch 15/15 | Loss: 1.0171

--- Training Moderate (Units: 80, LR: 0.005, Epochs: 30) ---
Epoch 6/30 | Loss: 0.5757
Epoch 12/30 | Loss: 0.1251
Epoch 18/30 | Loss: 0.0230
Epoch 24/30 | Loss: 0.0074
Epoch 30/30 | Loss: 0.0036

--- Training Aggressive (Units: 128, LR: 0.01, Epochs: 50) ---
Epoch 10/50 | Loss: 0.0010
Epoch 20/50 | Loss: 0.0001
Epoch 30/50 | Loss: 0.0000
Epoch 40/50 | Loss: 0.0000
Epoch 50/50 | Loss: 0.0000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Execution Complete! Downloaded 6 high-quality analytical figures for your report.
